# Load and Pre-process Data

In [11]:
import os
import numpy as np
import pandas as pd

DATA_PATH = 'nipstxt/'
print(os.listdir(DATA_PATH))

['nips04', 'nips07', 'nips05', 'nips10', 'nips02', 'MATLAB_NOTES', 'README_yann', 'idx', 'nips00', 'nips12', 'nips01', 'nips03', 'nips11', 'nips09', 'RAW_DATA_NOTES', 'nips06', 'orig', 'nips08']


In [12]:
folders = ["nips{0:02}".format(i) for i in range(0,13)]
# Read all texts into a list.
papers = []
for folder in folders:
    file_names = os.listdir(DATA_PATH + folder)
    for file_name in file_names:
        with open(DATA_PATH + folder + '/' + file_name, encoding='utf-8', errors='ignore', mode='r+') as f:
            data = f.read()
        papers.append(data)
len(papers)

1740

In [13]:
print(papers[0][:1000])

474 
OPTIMIZATION WITH ARTIFICIAL NEURAL NETWORK SYSTEMS: 
A MAPPING PRINCIPLE 
AND 
A COMPARISON TO GRADIENT BASED METHODS * 
Harrison MonFook Leong 
Research Institute for Advanced Computer Science 
NASA Ames Research Center 230-5 
Moffett Field, CA, 94035 
ABSTRACT 
General formulae for mapping optimization problems into systems of ordinary differential 
equations associated with artificial neural networks are presented. A comparison is made to optim- 
ization using gradient-search methods. The performance measure is the settling time from an initial 
state to a target state. A simple analytical example illustrates a situation where dynamical systems 
representing artificial neural network methods would settle faster than those representing gradient- 
search. Settling time was investigated for a more complicated optimization problem using com- 
puter simulations. The problem was a simplified version of a problem in medical imaging: deter- 
mining loci of cerebral activity from elect

## Basic Text Wrangling

In [14]:
%%time
import nltk

stop_words = nltk.corpus.stopwords.words('english')
wtk = nltk.tokenize.RegexpTokenizer(r'\w+')
wnl = nltk.stem.wordnet.WordNetLemmatizer()

def normalize_corpus(papers):
    norm_papers = []
    for paper in papers:
        paper = paper.lower()
        paper_tokens = [token.strip() for token in wtk.tokenize(paper)]
        paper_tokens = [wnl.lemmatize(token) for token in paper_tokens if not token.isnumeric()]
        paper_tokens = [token for token in paper_tokens if len(token) > 1]
        paper_tokens = [token for token in paper_tokens if token not in stop_words]
        paper_tokens = list(filter(None, paper_tokens))
        if paper_tokens:
            norm_papers.append(paper_tokens)
            
    return norm_papers
    
norm_papers = normalize_corpus(papers)
print(len(norm_papers))

1740
CPU times: user 9.24 s, sys: 12.5 ms, total: 9.25 s
Wall time: 9.25 s


# Text Representation with Feature Engineering

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(min_df=20, max_df=0.6, ngram_range=(1,2),
                     token_pattern=None, tokenizer=lambda doc: doc,
                     preprocessor=lambda doc: doc)
cv_features = cv.fit_transform(norm_papers)
cv_features.shape

(1740, 14412)

In [16]:
vocabulary = np.array(cv.get_feature_names_out())
print('Total Vocabulary Size:', len(vocabulary))

Total Vocabulary Size: 14412


# Topic Models with Latent Semantic Indexing (LSI)

In [17]:
%%time
from sklearn.decomposition import TruncatedSVD

TOTAL_TOPICS = 20

lsi_model = TruncatedSVD(n_components=TOTAL_TOPICS, n_iter=500, random_state=42)
document_topics = lsi_model.fit_transform(cv_features)

CPU times: user 1min 1s, sys: 3min 7s, total: 4min 9s
Wall time: 15.7 s


In [18]:
topic_terms = lsi_model.components_
topic_terms.shape

(20, 14412)

In [19]:
top_terms = 20
topic_key_term_idxs = np.argsort(-np.absolute(topic_terms), axis=1)[:, :top_terms]
topic_keyterm_weights = np.array([topic_terms[row, columns] 
                             for row, columns in list(zip(np.arange(TOTAL_TOPICS), topic_key_term_idxs))])
topic_keyterms = vocabulary[topic_key_term_idxs]
topic_keyterms_weights = list(zip(topic_keyterms, topic_keyterm_weights))
for n in range(TOTAL_TOPICS):
    print('Topic #'+str(n+1)+':')
    print('='*50)
    d1 = []
    d2 = []
    terms, weights = topic_keyterms_weights[n]
    term_weights = sorted([(t, w) for t, w in zip(terms, weights)], 
                          key=lambda row: -abs(row[1]))
    for term, wt in term_weights:
        if wt >= 0:
            d1.append((term, round(wt, 3)))
        else:
            d2.append((term, round(wt, 3)))

    print('Direction 1:', d1)
    print('-'*50)
    print('Direction 2:', d2)
    print('-'*50)
    print()

Topic #1:
Direction 1: [('state', 0.221), ('neuron', 0.169), ('image', 0.138), ('cell', 0.13), ('layer', 0.13), ('feature', 0.127), ('probability', 0.121), ('hidden', 0.114), ('distribution', 0.105), ('rate', 0.098), ('signal', 0.095), ('task', 0.093), ('class', 0.092), ('noise', 0.09), ('net', 0.089), ('recognition', 0.089), ('representation', 0.088), ('field', 0.082), ('rule', 0.082), ('step', 0.08)]
--------------------------------------------------
Direction 2: []
--------------------------------------------------

Topic #2:
Direction 1: [('cell', 0.417), ('neuron', 0.39), ('response', 0.175), ('stimulus', 0.155), ('visual', 0.131), ('spike', 0.13), ('firing', 0.117), ('synaptic', 0.11), ('activity', 0.104), ('cortex', 0.097), ('field', 0.085), ('frequency', 0.085), ('direction', 0.082), ('circuit', 0.082), ('motion', 0.082)]
--------------------------------------------------
Direction 2: [('state', -0.289), ('probability', -0.109), ('hidden', -0.098), ('class', -0.091), ('policy',

In [20]:
dt_df = pd.DataFrame(np.round(document_topics, 3), 
                     columns=['T'+str(i) for i in range(1, TOTAL_TOPICS+1)])
dt_df.T

,0,1,2,3,4,5,6,7,8,9,...,1730,1731,1732,1733,1734,1735,1736,1737,1738,1739
T1,46.460,16.409,30.983,53.568,25.402,50.264,26.404,47.965,37.996,29.326,...,31.067,29.219,29.430,24.808,31.315,31.164,26.190,42.274,22.223,51.954
T2,-8.457,2.372,1.361,-33.247,-5.809,-0.977,-3.343,7.450,0.651,-2.076,...,-8.154,-13.145,-15.920,-2.138,8.733,-17.652,-6.170,19.998,-5.567,-47.574
T3,12.225,5.192,-15.406,66.925,3.305,9.111,-10.793,-34.564,0.843,0.066,...,-3.265,-8.413,-9.257,-11.552,-4.152,17.795,-18.192,-22.877,-5.495,74.045
T4,-1.710,-5.582,12.508,34.736,-6.951,-15.598,-2.241,36.673,-1.918,-7.140,...,-7.144,-16.734,-21.758,11.604,-0.227,-4.464,1.512,15.477,-2.061,38.714
T5,-10.600,-1.503,-12.688,-1.941,6.595,13.568,22.357,-10.156,12.997,4.699,...,1.583,-17.648,-20.681,-8.468,8.995,-8.530,-1.020,-29.031,-1.928,-9.831
T6,-8.041,-5.476,-13.966,1.575,1.415,-10.782,6.133,-29.757,-5.617,-8.583,...,4.986,-1.002,1.635,-3.801,2.048,-5.295,1.393,7.662,-2.638,1.263
T7,-12.082,-1.622,-5.677,8.203,-13.508,-3.706,11.440,0.884,-18.846,-5.382,...,-3.197,-0.796,6.670,3.425,6.505,8.219,2.511,11.083,1.306,14.966
T8,4.076,0.020,3.066,-8.679,2.272,7.646,5.586,-9.254,-7.572,-1.441,...,-0.028,-9.812,-6.009,8.839,13.737,-2.624,-11.047,-3.619,5.220,-16.090
T9,-6.761,-2.974,-2.528,1.103,3.927,7.051,-1.330,8.700,-13.374,-6.764,...,-4.348,-1.269,4.545,-1.995,-1.068,5.231,2.005,9.657,-2.873,20.386
T10,2.848,-6.448,4.497,-1.155,-1.232,10.947,0.264,4.210,8.069,11.700,...,-8.572,-7.287,0.150,3.148,6.959,12.561,-15.313,14.018,-0.376,17.066


In [21]:
document_numbers = [13, 250, 500]

for document_number in document_numbers:
    top_topics = list(dt_df.columns[np.argsort(-np.absolute(dt_df.iloc[document_number].values))[:3]])
    print('Document #'+str(document_number)+':')
    print('Dominant Topics (top 3):', top_topics)
    print('Paper Summary:')
    print(papers[document_number][:500])
    print()

Document #13:
Dominant Topics (top 3): ['T2', 'T1', 'T6']
Paper Summary:
701 
DISCOVERING STRUCTURE FROM MOTION IN 
MONKEY, MAN AND MACHINE 
Ralph M. Siegel* 
The Salk Institute of Biology, La Jolla, Ca. 92037 
ABSTRACT 
The ability to obtain three-dimensional structure from visual motion is 
important for survival of human and non-human primates. Using a parallel process- 
ing model, the current work explores how the biological visual system might solve 
this problem and how the neurophysiologist might go about understanding the 
solution. 
INTRODUCTION 
Psychophysi

Document #250:
Dominant Topics (top 3): ['T1', 'T5', 'T12']
Paper Summary:
364 Jain and Waibel 
Incremental Parsing by Modular Recurrent 
Connectionist Networks 
Ajay N. Jain Alex H. Waibel 
School of Computer Science 
Carnegie Mellon University 
Pittsburgh, PA 15213 
ABSTRACT 
We present a novel, modular, recurrent connectionist network architec- 
ture which learns to robustly perform incremental parsing of complex 
sent

# Topic Models with Latent Dirichlet Allocation (LDA)

In [22]:
%%time
from sklearn.decomposition import LatentDirichletAllocation

lda_model = LatentDirichletAllocation(n_components =TOTAL_TOPICS, max_iter=500, max_doc_update_iter=50,
                                      learning_method='online', batch_size=1740, learning_offset=50., 
                                      random_state=42, n_jobs=16)
document_topics = lda_model.fit_transform(cv_features)

CPU times: user 11 s, sys: 4.87 s, total: 15.9 s
Wall time: 54.6 s


In [23]:
topic_terms = lda_model.components_

In [26]:
topic_key_term_idxs = np.argsort(-np.absolute(topic_terms), axis=1)[:, :top_terms]
topic_keyterms = vocabulary[topic_key_term_idxs]
topics = [', '.join(topic) for topic in topic_keyterms]
pd.set_option('display.max_colwidth', None)
topics_df = pd.DataFrame(topics,
                         columns = ['Terms per Topic'],
                         index=['Topic'+str(t) for t in range(1, TOTAL_TOPICS+1)])
topics_df

,Terms per Topic
Topic1,"state, image, neuron, feature, recognition, object, net, hidden, structure, distribution, variable, probability, equation, node, dynamic, layer, estimate, local, target, et"
Topic2,"image, neuron, class, state, net, cell, probability, control, recognition, step, feature, matrix, noise, hidden, distribution, signal, layer, size, dynamic, node"
Topic3,"layer, image, feature, state, neuron, signal, rate, line, distribution, response, task, field, test, recognition, visual, structure, target, representation, hidden, cell"
Topic4,"distribution, probability, class, gaussian, variable, estimate, sample, density, mixture, likelihood, test, approximation, matrix, prior, log, prediction, variance, feature, noise, classification"
Topic5,"neuron, state, image, distribution, control, cell, test, feature, speech, field, signal, task, noise, probability, net, recognition, response, theory, dynamic, hidden"
Topic6,"control, object, expert, motor, trajectory, controller, movement, position, module, dynamic, robot, task, arm, forward, view, hand, inverse, architecture, target, command"
Topic7,"state, task, control, layer, image, neuron, classifier, probability, hidden, level, et, distribution, test, object, rate, component, variable, response, dynamic, size"
Topic8,"cell, state, neuron, image, layer, rule, hidden, head, rate, feature, recognition, direction, probability, task, hidden unit, class, field, sample, net, equation"
Topic9,"cell, layer, feature, representation, field, rule, node, et al, direction, state, neuron, place, probability, map, rate, noise, et, hidden, net, variable"
Topic10,"recognition, hidden, layer, feature, representation, word, speech, task, net, trained, architecture, hidden unit, test, node, image, sequence, level, rule, classification, experiment"


In [27]:
pd.options.display.float_format = '{:,.3f}'.format
dt_df = pd.DataFrame(document_topics, 
                     columns=['T'+str(i) for i in range(1, TOTAL_TOPICS+1)])
dt_df.T

,0,1,2,3,4,5,6,7,8,9,...,1730,1731,1732,1733,1734,1735,1736,1737,1738,1739
T1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
T2,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
T3,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
T4,0.140,0.010,0.305,0.000,0.035,0.032,0.038,0.156,0.004,0.000,...,0.323,0.910,0.994,0.341,0.000,0.049,0.832,0.467,0.312,0.080
T5,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
T6,0.013,0.000,0.000,0.017,0.000,0.000,0.000,0.043,0.018,0.000,...,0.000,0.010,0.000,0.081,0.000,0.000,0.000,0.000,0.012,0.000
T7,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
T8,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
T9,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
T10,0.001,0.000,0.042,0.000,0.166,0.062,0.837,0.179,0.239,0.062,...,0.354,0.000,0.000,0.257,0.200,0.028,0.167,0.000,0.419,0.019


In [28]:
pd.options.display.float_format = '{:,.5f}'.format
pd.set_option('display.max_colwidth', 200)

max_contrib_topics = dt_df.max(axis=0)
dominant_topics = max_contrib_topics.index
contrib_perc = max_contrib_topics.values
document_numbers = [dt_df[dt_df[t] == max_contrib_topics.loc[t]].index[0]
                       for t in dominant_topics]
documents = [papers[i] for i in document_numbers]

results_df = pd.DataFrame({'Dominant Topic': dominant_topics, 'Contribution %': contrib_perc,
                          'Paper Num': document_numbers, 'Topic': topics_df['Terms per Topic'], 
                          'Paper Name': documents})
results_df

,Dominant Topic,Contribution %,Paper Num,Topic,Paper Name
Topic1,T1,0.00033,125,"state, image, neuron, feature, recognition, object, net, hidden, structure, distribution, variable, probability, equation, node, dynamic, layer, estimate, local, target, et",794 \nNEURAL ARCHITECTURE \nValentino Braitenberg \nMax Planck Institute \nFederal Republic of Germany \nABSTRACT\nWhile we are waiting for the ultimate biophysics of cell membranes and synapses \...
Topic2,T2,0.00033,125,"image, neuron, class, state, net, cell, probability, control, recognition, step, feature, matrix, noise, hidden, distribution, signal, layer, size, dynamic, node",794 \nNEURAL ARCHITECTURE \nValentino Braitenberg \nMax Planck Institute \nFederal Republic of Germany \nABSTRACT\nWhile we are waiting for the ultimate biophysics of cell membranes and synapses \...
Topic3,T3,0.00033,125,"layer, image, feature, state, neuron, signal, rate, line, distribution, response, task, field, test, recognition, visual, structure, target, representation, hidden, cell",794 \nNEURAL ARCHITECTURE \nValentino Braitenberg \nMax Planck Institute \nFederal Republic of Germany \nABSTRACT\nWhile we are waiting for the ultimate biophysics of cell membranes and synapses \...
Topic4,T4,0.99942,1727,"distribution, probability, class, gaussian, variable, estimate, sample, density, mixture, likelihood, test, approximation, matrix, prior, log, prediction, variance, feature, noise, classification","Bayesian model selection for Support \nVector machines, Gaussian processes and \nother kernel classifiers \nMatthias Seeger \nInstitute for Adaptive and Neural Computation \nUniversity of Edinburg..."
Topic5,T5,0.00033,125,"neuron, state, image, distribution, control, cell, test, feature, speech, field, signal, task, noise, probability, net, recognition, response, theory, dynamic, hidden",794 \nNEURAL ARCHITECTURE \nValentino Braitenberg \nMax Planck Institute \nFederal Republic of Germany \nABSTRACT\nWhile we are waiting for the ultimate biophysics of cell membranes and synapses \...
Topic6,T6,0.99123,444,"control, object, expert, motor, trajectory, controller, movement, position, module, dynamic, robot, task, arm, forward, view, hand, inverse, architecture, target, command","Recognition of Manipulated Objects \nby Motor Learning \nHiroaki Gomi Mitsuo Kawato \nATR Auditory and Visual Perception Research Laboratories, \nInui-dani, Sanpei-dani, Seika-cho, Soraku-gun, Kyo..."
Topic7,T7,0.00033,125,"state, task, control, layer, image, neuron, classifier, probability, hidden, level, et, distribution, test, object, rate, component, variable, response, dynamic, size",794 \nNEURAL ARCHITECTURE \nValentino Braitenberg \nMax Planck Institute \nFederal Republic of Germany \nABSTRACT\nWhile we are waiting for the ultimate biophysics of cell membranes and synapses \...
Topic8,T8,0.00033,125,"cell, state, neuron, image, layer, rule, hidden, head, rate, feature, recognition, direction, probability, task, hidden unit, class, field, sample, net, equation",794 \nNEURAL ARCHITECTURE \nValentino Braitenberg \nMax Planck Institute \nFederal Republic of Germany \nABSTRACT\nWhile we are waiting for the ultimate biophysics of cell membranes and synapses \...
Topic9,T9,0.00033,125,"cell, layer, feature, representation, field, rule, node, et al, direction, state, neuron, place, probability, map, rate, noise, et, hidden, net, variable",794 \nNEURAL ARCHITECTURE \nValentino Braitenberg \nMax Planck Institute \nFederal Republic of Germany \nABSTRACT\nWhile we are waiting for the ultimate biophysics of cell membranes and synapses \...
Topic10,T10,0.99947,420,"recognition, hidden, layer, feature, representation, word, speech, task, net, trained, architecture, hidden unit, test, node, image, sequence, level, rule, classification, experiment","A Recurrent Neural Network for Word Identification \nfrom Continuous Phoneme Strings \nRobert B. Allen \nBellcore \nMorristown, NJ 07962-1910 \nCandace A. Kamm \nBellcore \nMorristown, NJ 0

# Topic Models with Non-Negative Matrix Factorization (NMF)

In [44]:
%%time
from sklearn.decomposition import NMF

nmf_model = NMF(n_components=TOTAL_TOPICS, solver='cd', max_iter=500,
                random_state=42, l1_ratio=.85)
document_topics = nmf_model.fit_transform(cv_features)

CPU times: user 2.37 s, sys: 3.68 s, total: 6.04 s
Wall time: 1.34 s


In [45]:
topic_terms = nmf_model.components_
topic_key_term_idxs = np.argsort(-np.absolute(topic_terms), axis=1)[:, :top_terms]
topic_keyterms = vocabulary[topic_key_term_idxs]
topics = [', '.join(topic) for topic in topic_keyterms]
pd.set_option('display.max_colwidth', None)
topics_df = pd.DataFrame(topics,
                         columns = ['Terms per Topic'],
                         index=['Topic'+str(t) for t in range(1, TOTAL_TOPICS+1)])
topics_df

,Terms per Topic
Topic1,"bound, class, threshold, theorem, let, probability, size, dimension, vc, sample, polynomial, distribution, proof, net, complexity, approximation, theory, loss, xi, vc dimension"
Topic2,"neuron, synaptic, connection, potential, dynamic, activity, synapsis, excitatory, layer, simulation, synapse, inhibitory, delay, biological, state, et, equation, et al, activation, fig"
Topic3,"state, action, policy, step, reinforcement, optimal, reinforcement learning, transition, probability, reward, value function, dynamic, markov, machine, task, agent, finite, iteration, sequence, decision"
Topic4,"image, face, pixel, recognition, local, scale, texture, digit, distance, filter, scene, vision, edge, facial, pca, representation, region, visual, surface, database"
Topic5,"hidden, layer, net, hidden unit, task, hidden layer, architecture, back, trained, propagation, connection, back propagation, activation, representation, output unit, neural net, internal, generalization, training set, learn"
Topic6,"cell, firing, head, response, direction, rat, layer, cortex, activity, ii, spatial, synaptic, inhibitory, synapsis, simulation, cue, region, property, complex, lot"
Topic7,"word, recognition, speech, context, hmm, speaker, speech recognition, character, phoneme, probability, frame, sequence, rate, level, test, acoustic, experiment, letter, segmentation, state"
Topic8,"signal, noise, source, filter, frequency, component, speech, channel, sound, independent, separation, ica, phase, auditory, eeg, matrix, blind, delay, acoustic, spectrum"
Topic9,"control, controller, trajectory, motor, movement, task, dynamic, forward, feedback, arm, inverse, position, robot, architecture, hand, force, target, change, command, adaptive"
Topic10,"circuit, chip, current, analog, voltage, vlsi, transistor, gate, pulse, threshold, design, implementation, synapse, bit, digital, device, analog vlsi, cmos, pp, line"


In [46]:
pd.options.display.float_format = '{:,.3f}'.format
dt_df = pd.DataFrame(document_topics, 
                     columns=['T'+str(i) for i in range(1, TOTAL_TOPICS+1)])
dt_df.head(10)

,T1,T2,T3,T4,T5,T6,T7,T8,T9,T10,T11,T12,T13,T14,T15,T16,T17,T18,T19,T20
0,0.004,0.057,0.355,0.000,0.000,0.010,0.000,0.008,0.028,0.158,0.000,0.031,0.042,0.075,0.000,0.072,0.264,0.026,0.288,1.412
1,0.019,0.148,0.032,0.000,0.000,0.000,0.017,0.000,0.041,0.001,0.115,0.002,0.000,0.000,0.000,0.000,0.032,0.016,0.029,0.538
2,0.000,0.000,0.015,0.555,0.037,0.009,0.000,0.091,0.000,0.035,0.000,0.007,0.013,0.009,0.000,0.042,0.252,0.100,0.000,0.679
3,0.000,0.012,1.947,0.001,0.000,0.021,0.000,0.028,0.156,0.054,0.138,0.264,0.001,0.004,0.110,0.000,0.077,0.000,0.061,0.050
4,0.013,0.042,0.094,0.000,0.249,0.000,0.000,0.000,0.000,0.017,0.023,0.319,0.000,0.000,0.000,0.000,0.000,0.000,0.074,0.560
5,0.080,0.248,0.127,0.000,0.147,0.000,0.111,0.439,0.000,0.000,0.000,0.066,0.000,0.000,0.000,0.168,0.000,0.000,1.781,0.191
6,0.023,0.118,0.000,0.020,0.181,0.000,0.590,0.077,0.000,0.011,0.000,0.013,0.023,0.073,0.268,0.000,0.034,0.074,0.125,0.000
7,0.000,0.000,0.000,0.915,0.000,0.001,0.000,0.171,0.000,0.000,0.020,0.000,0.000,0.000,0.011,0.056,0.010,1.633,1.937,0.005
8,0.007,0.046,0.000,0.015,0.108,0.000,0.000,0.000,0.165,0.288,0.017,0.019,0.450,0.071,0.000,0.000,0.000,0.000,1.477,0.264
9,0.047,0.000,0.000,0.012,0.000,0.000,0.000,0.185,0.000,0.084,0.000,0.000,0.113,0.000,0.000,0.000,0.000,0.000,1.836,0.225


In [47]:
pd.options.display.float_format = '{:,.5f}'.format
pd.set_option('display.max_colwidth', 200)

max_score_topics = dt_df.max(axis=0)
dominant_topics = max_score_topics.index
term_score = max_score_topics.values
document_numbers = [dt_df[dt_df[t] == max_score_topics.loc[t]].index[0]
                       for t in dominant_topics]
documents = [papers[i] for i in document_numbers]

results_df = pd.DataFrame({'Dominant Topic': dominant_topics, 'Max Score': term_score,
                          'Paper Num': document_numbers, 'Topic': topics_df['Terms per Topic'], 
                          'Paper Name': documents})
results_df

,Dominant Topic,Max Score,Paper Num,Topic,Paper Name
Topic1,T1,0.50902,1274,"bound, class, threshold, theorem, let, probability, size, dimension, vc, sample, polynomial, distribution, proof, net, complexity, approximation, theory, loss, xi, vc dimension","For valid generalization, the size of the \nweights is more important than the size \nof the network \nPeter L. Bartlett \nDepartment of Systems Engineering \nResearch School of Information Scienc..."
Topic2,T2,1.62499,406,"neuron, synaptic, connection, potential, dynamic, activity, synapsis, excitatory, layer, simulation, synapse, inhibitory, delay, biological, state, et, equation, et al, activation, fig","Signal Processing by Multiplexing and \nDemultiplexing in Neurons \nDavid C. Tam \nDivision of Neuroscience \nBaylor College of Medicine \nHouston, TX 77030 \ndtamCnext-cns.neusc.bcm.tmc.edu \nAb..."
Topic3,T3,3.15877,1175,"state, action, policy, step, reinforcement, optimal, reinforcement learning, transition, probability, reward, value function, dynamic, markov, machine, task, agent, finite, iteration, sequence, de...","Reinforcement Learning for Mixed \nOpen-loop and Closed-loop Control \nEric A. Hansen, Andrew G. Barto, and Shlomo Zilbersteln \nDepartment of Computer Science \nUniversity of Massachusetts \nAmhe..."
Topic4,T4,1.76966,1614,"image, face, pixel, recognition, local, scale, texture, digit, distance, filter, scene, vision, edge, facial, pca, representation, region, visual, surface, database",Image representations for facial expression \ncoding \nMarian Stewart Bartlett* \nU.C. San Diego \nmarnisalk. edu \nJavier R. Movellan \nU.C. San Diego \nmovellancogsc. ucsd. edu \nPaul Ekman \n...
Topic5,T5,1.10328,795,"hidden, layer, net, hidden unit, task, hidden layer, architecture, back, trained, propagation, connection, back propagation, activation, representation, output unit, neural net, internal, generali...","Generation of Internal Representation \nby c-Transformation \nRyotaro Kamimura \nInformation Science Laboratory \nTokai University \n1117 Kitakaname Hiratsuka Kanagawa 259-12, Japan \nAbstract \nI..."
Topic6,T6,4.35568,34,"cell, firing, head, response, direction, rat, layer, cortex, activity, ii, spatial, synaptic, inhibitory, synapsis, simulation, cue, region, property, complex, lot","317 \nPARTITIONING OF SENSORY DATA BY A COPTICAI, NETWOPK  \nRichard Granger, Jos Ambros-Ingerson, Howard Henry, Gary Lynch \nCenter for the Neurobiology of Learning and Memory \nUniversity of..."
Topic7,T7,2.45285,1400,"word, recognition, speech, context, hmm, speaker, speech recognition, character, phoneme, probability, frame, sequence, rate, level, test, acoustic, experiment, letter, segmentation, state","Comparison of Human and Machine Word \nRecognition \nM. Schenkel \nDept of Electrical Eng. \nUniversity of Sydney \nSydney, NSW 2006, Australia \nschenkel@sedal.usyd.edu.au \nC. Latimer \nDept of ..."
Topic8,T8,1.34497,284,"signal, noise, source, filter, frequency, component, speech, channel, sound, independent, separation, ica, phase, auditory, eeg, matrix, blind, delay, acoustic, spectrum","232 Sejnowski, Yuhas, Goldstein and Jenkins \nCombining Visual and \nwith a Neural Network \nAcoustic Speech Signals \nImproves Intelligibility \nT.J. Sejnowski \nThe Salk Institute \nand \nDepart..."
Topic9,T9,1.78388,966,"control, controller, trajectory, motor, movement, task, dynamic, forward, feedback, arm, inverse, position, robot, architecture, hand, force, target, change, command, adaptive","An Integrated Architecture of Adaptive Neural Network \nControl for Dynamic Systems \nLiu Ke '2 Robert L. Tokaf Brian D.McVey z \nCenter for Nonlinear Studies, 2Applied Theoretical Physics Divis..."
Topic10,T10,1.39558,1647,"circuit, chip, current, analog, voltage, vlsi, transistor, gate, pulse, threshold, design, implementation, synapse, bit, digital, device, analog vlsi, cmos, pp, line","Kirchoff Law Markov Fields for Analog \nCircuit Design \nRichard M. Golden * \nRMG Consultin

# Predicting Topics for New Research Papers

In [48]:
import glob
# papers manually downloaded from NIPS 16
# https://papers.nips.cc/book/advances-in-neural-information-processing-systems-29-2016

new_paper_files = glob.glob('./test_data/nips16*.txt')
new_papers = []
for fn in new_paper_files:
    with open(fn, encoding='utf-8', errors='ignore', mode='r+') as f:
        data = f.read()
        new_papers.append(data)
              
print('Total New Papers:', len(new_papers))

Total New Papers: 4


In [49]:
norm_new_papers = normalize_corpus(new_papers)
cv_new_features = cv.transform(norm_new_papers)
cv_new_features.shape

(4, 14412)

In [50]:
topic_predictions = nmf_model.transform(cv_new_features)
best_topics = [[(topic, round(sc, 3)) 
                    for topic, sc in sorted(enumerate(topic_predictions[i]), 
                                            key=lambda row: -row[1])[:2]] 
                        for i in range(len(topic_predictions))]
best_topics

[[(15, 0.615), (19, 0.371)],
 [(3, 0.961), (1, 0.608)],
 [(2, 2.246), (15, 0.282)],
 [(3, 1.383), (6, 1.088)]]

In [51]:
results_df = pd.DataFrame()
results_df['Papers'] = range(1, len(new_papers)+1)
results_df['Dominant Topics'] = [[topic_num+1 for topic_num, sc in item] for item in best_topics]
res = results_df.set_index(['Papers'])['Dominant Topics'].apply(pd.Series).stack().reset_index(level=1, drop=True)
results_df = pd.DataFrame({'Dominant Topics': res.values}, index=res.index)
results_df['Topic Score'] = [topic_sc for topic_list in 
                                        [[round(sc*100, 2) 
                                              for topic_num, sc in item] 
                                                 for item in best_topics] 
                                    for topic_sc in topic_list]

results_df['Topic Desc'] = [topics_df.iloc[t-1]['Terms per Topic'] for t in results_df['Dominant Topics'].values]
results_df['Paper Desc'] = [new_papers[i-1][:200] for i in results_df.index.values]

results_df

,Dominant Topics,Topic Score,Topic Desc,Paper Desc
Papers,,,,
1,16,61.50000,"distribution, gaussian, probability, mixture, variable, density, likelihood, prior, bayesian, component, posterior, em, log, estimate, sample, approximation, estimation, conditional, structure, ma...","Cooperative Graphical Models\nJosip Djolonga\nDept. of Computer Science, ETH Zurich ¨\njosipd@inf.ethz.ch\nStefanie Jegelka\nCSAIL, MIT\nstefje@mit.edu\nSebastian Tschiatschek\nDept. of Computer S..."
1,20,37.10000,"equation, gradient, solution, matrix, optimal, generalization, rate, local, minimum, distance, line, convergence, noise, optimization, training set, constraint, descent, average, cost, test","Cooperative Graphical Models\nJosip Djolonga\nDept. of Computer Science, ETH Zurich ¨\njosipd@inf.ethz.ch\nStefanie Jegelka\nCSAIL, MIT\nstefje@mit.edu\nSebastian Tschiatschek\nDept. of Computer S..."
2,4,96.10000,"image, face, pixel, recognition, local, scale, texture, digit, distance, filter, scene, vision, edge, facial, pca, representation, region, visual, surface, database","Automated scalable segmentation of neurons from\nmultispectral images\nUygar Sümbül\nGrossman Center for the Statistics of Mind\nand Dept. of Statistics, Columbia University\nDouglas Roossien Jr.\..."
2,2,60.80000,"neuron, synaptic, connection, potential, dynamic, activity, synapsis, excitatory, layer, simulation, synapse, inhibitory, delay, biological, state, et, equation, et al, activation, fig","Automated scalable segmentation of neurons from\nmultispectral images\nUygar Sümbül\nGrossman Center for the Statistics of Mind\nand Dept. of Statistics, Columbia University\nDouglas Roossien Jr.\..."
3,3,224.60000,"state, action, policy, step, reinforcement, optimal, reinforcement learning, transition, probability, reward, value function, dynamic, markov, machine, task, agent, finite, iteration, sequence, de...","PAC Reinforcement Learning with Rich Observations\nAkshay Krishnamurthy\nUniversity of Massachusetts, Amherst\nAmherst, MA, 01003\nakshay@cs.umass.edu\nAlekh Agarwal\nMicrosoft Research\nNew York,..."
3,16,28.20000,"distribution, gaussian, probability, mixture, variable, density, likelihood, prior, bayesian, component, posterior, em, log, estimate, sample, approximation, estimation, conditional, structure, ma...","PAC Reinforcement Learning with Rich Observations\nAkshay Krishnamurthy\nUniversity of Massachusetts, Amherst\nAmherst, MA, 01003\nakshay@cs.umass.edu\nAlekh Agarwal\nMicrosoft Research\nNew York,..."
4,4,138.30000,"image, face, pixel, recognition, local, scale, texture, digit, distance, filter, scene, vision, edge, facial, pca, representation, region, visual, surface, database","Unsupervised Learning of Spoken Language with\nVisual Context\nDavid Harwath, Antonio Torralba, and James R. Glass\nComputer Science and Artificial Intelligence Laboratory\nMassachusetts Institute..."
4,7,108.80000,"word, recognition, speech, context, hmm, speaker, speech recognition, character, phoneme, probability, frame, sequence, rate, level, test, acoustic, experiment, letter, segmentation, state","Unsupervised Learning of Spoken Language with\nVisual Context\nDavid Harwath, Antonio Torralba, and James R. Glass\nComputer Science and Artificial Intelligence Laboratory\nMassachusetts Institute..."


# Persisting Model and Transformers

### This is just for visualizing the topics in the other notebook (since PyLDAViz expands the notebook size)

In [53]:
import dill

with open('nmf_model.pkl', 'wb') as f:
    dill.dump(nmf_model, f)
with open('cv_features.pkl', 'wb') as f:
    dill.dump(cv_features, f)
with open('cv.pkl', 'wb') as f:
    dill.dump(cv, f)